In [ ]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain-community langchain-openai
%pip install -Uq python_dotenv

In [ ]:
import json
from typing import List

#Unstructured for document parsing

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

#Langchain component

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import  load_dotenv

load_dotenv()

In [ ]:
#the partion method the first method to partion our pdf into atomic element

def partition_document(file_path:str):
    """"Extract Element from the PDF using Unstrcutured"""

    print(f"Partitioning Document.... {file_path}")

    element = partition_pdf(
            filename=file_path, #our Pdf file path
            strategy="hi_res", # use most accurate (but slower) processing method of extraction
            infer_table_structure=True, #keep Table as structured Html, not jambed text
            extract_image_block_types=['Image'], # Grab images found in the pdf
            extract_image_block_to_payload=True # store images as based64 data that u can actively use
    ) 
    print(f"Extracted {len(element)} elments")
    return element

#testing with our pdf file
file_path = "../docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path=file_path)



In [ ]:
elements[31].to_dict()

In [ ]:
#See What image elements looks-like
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

In [ ]:
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[1].to_dict()

In [ ]:
#chunking by title

def create_chunk_by_title(elements):
    """Create intelligent chunks title title-baded"""
    print("Creating Smart Chunks....")

    chunks = chunk_by_title(
        elements,
        max_characters=3000, #Hard limit, this limit is never exceeded
        new_after_n_chars=2400, #Try to start a new chunk after 2400 x-ters
        combine_text_under_n_chars=500 #combine  tiny chunks under 500 chars with neightbours
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

In [ ]:
chunks = create_chunk_by_title(elements)

In [ ]:
chunks[11].metadata.orig_elements[-1].to_dict()

In [ ]:
def separate_content_types(chunk):
    """Analyse what type of content are in the chunk"""

    content_data = {
        'text':chunk.text,
        'tables':[],
        'images':[],
        'types' :['text']
    }

    #check for table and images in the original elements

    if hasattr(chunk,'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__


            #Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            #handle images

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)


    content_data['types'] = list(set(content_data['types']))
    return content_data

In [ ]:
def create_ai_enhanced_summary(text:str,tables:List[str], images:List[str])->str:
    """Create AI-enhanced sumary for mixed content"""

    try:
        #Initialise LLM (need vision model for images)

        llm = ChatOpenAI(model="gpt-4o", temperature=0)

        #Build the context prompt

        prompt_text = f""" You are Creating a searchable description for document content retrieval

        CONTEXT TO ANALYSE:
        TEXT CONTENT/
        {text}
            \n\n
        """
        prompt_text +=""" Your Task :
                               Generate a comprehensive, searchable description that covers:
                               1.key facts, numbers, and data points from text and table
                               2.Main topics and concepts discoused
                               3.questions this content could answer
                               4.visual content analysis(charts, diagrams, patterns in images)
                               5.Alternative search terms users might use
               
                               make it detailed and searchable - findability over bravity.
               
                               Searchable Description description: \n\n
                                 """

        #add tables if present
        if tables:
            prompt_text += "TABLE\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n {table}"

        #build message content starting with text
        message_content = [{"type":"text", "text":prompt_text}]

        #add images to the message

        for image_base64 in images:

            message_content.append({"type":"image_url",
                                    "image_url":{"url": f"data:image/jpeg;base64,{image_base64}"}}) 

            #Send to AI and get response
        message = HumanMessage(content=message_content)
        reponse = llm.invoke([message])
        return reponse.content
    except Exception as e:
        print(f"AI Samary failed: {e}")
        #Fallback to simple sumary
        sumary = f"{text[:300]}..."
        if tables:
            sumary += f"[Contains {len(tables)} tales(s)]"
        if images:
            sumary += f"[Contains {len(images)} images(s)]"
        return sumary


In [ ]:
def sumerise_chunks(chunks):
    """Process all chunks with AI Sumeries"""
    print("Processing Chunks with AI Sumeries")


    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i +1
        print(f"Processing chunk {current_chunk} / {total_chunks}")

        #analyse chunk content

        content_data = separate_content_types(chunk)

        #Debug print

        print(f"Types found: {content_data['types']}")
        print(f"Tables: {len(content_data['tables'])}, Images :{len(content_data['images'])})")

        #create ai-enhanced sumary

        if content_data['tables'] or content_data['images']:
            print(f"Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f" AI Summary Created successfulle")
                print(f"Enhanced Content preview: {enhanced_content[:200]}")
            except Exception as e:
                print(f"AI Sumary failed: {e}")
                enhanced_content=content_data['text']
        else:
            print(f"Using raw text (no table/images)")
            enhanced_content=content_data['text']
        #Create Langchain documents with metadata
        doc=Document(
            page_content=enhanced_content,
            metadata={
                "source": chunk.metadata.filename,             
                "page_number": chunk.metadata.page_number or 0, 
                "chunk_index": i,                               
                "types": ",".join(content_data["types"]),      
                "has_summary": bool(content_data["tables"] or content_data["images"]),
                "original_content":json.dumps({
                    "raw_text":content_data['text'],
                    "tables_html":content_data["tables"],
                    "images_base64":content_data['images']
                })
            }
        )

        langchain_documents.append(doc)

    print(f"Processed {len(langchain_documents)}")
    return langchain_documents
processed_chunks=sumerise_chunks(chunks)
print(processed_chunks[0].metadata["source"], processed_chunks[0].metadata["page_number"])

In [ ]:
load_dotenv()
def create_vector_store(documents,persistent_directory="../dbv1/chroma_db"):
    """ Create and persist ChromaDB vector store"""

    print("Creating embedding and storing in chroma")

    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    #Creating Chroma vector store

    print("--- Creating vector store---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persistent_directory,
        collection_metadata={"hnsw:space":"cosine"}
    )

    print(f"Finished Creating vectorestore")

    print(f"Vector Store created and saved to {persistent_directory}")
    return vectorstore
#create the vector store
db=create_vector_store(processed_chunks)

In [ ]:
#Query the vector store

chat_history = []
def generate_final_answer(user_question):
    """Generate final answer using multimodal content"""
    #creating the chat history list
   

  

    try:
        #Initailize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)


        if chat_history:
                message = [
                    SystemMessage(content="Given the chat History, rewrite the new question question to be standalon and search")
                ] + chat_history + [
                    HumanMessage(content=f" New question{user_question}")
                ]
                result = llm.invoke(message)
                search_question = result.content.strip()
                print(f"Searching for:{search_question}")
        else:
                search_question = user_question

        #Step 2: Find relevant documents
        retriever = db.as_retriever(search_kwargs={"k":3})
        chunks = retriever.invoke(search_question)

        #build the text prompt

        prompt_text = f"""Based on the following documents, please answer this question: {search_question}

CONTENT TO ANALYZE:"""
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1}---"
            if 'original_content' in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                #add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"Text: \n {raw_text}\n\n"
                table_html = original_data.get("tables_html", [])
                if table_html:
                    prompt_text += f"TABLES: \n"
                    for j, table in enumerate(table_html):
                        prompt_text += f"Table {j+1}: \n {table}"
            prompt_text+="\n"
        prompt_text += """
        Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

        ANSWER:"""

        #build all images from all chunks
        message_content =  [{"type":"text", "text":prompt_text}]

        for i, chunk in enumerate(chunks):
            if 'original_content' in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])

                images_base64 = original_data.get('images_base64', [])
                for image_base64 in images_base64:
                    message_content.append({
                        "type":"image_url",
                        "image_url":{"url": f"data:image/jpeg;base64,{image_base64}"}
                    })

        #Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        chat_history.append(HumanMessage(content=user_question))
        chat_history.append(AIMessage(content=response.content))

        return response.content
    except Exception as e:
        print(f"Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."



In [ ]:

    #simple chat loop
def start_chat():
    print("Ask me any question!! type quit to exit")

    while True:
        question = input("\n Your question :")
        if question.lower() == 'quit':
            print("GoodBye")
            break
        ans_r=generate_final_answer(question)
        print(f"Model reponse: {ans_r}")


In [ ]:
start_chat()